In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Data
tf = transforms.Compose([
    transforms.ToTensor()
])

data = datasets.MNIST("data", train=True, download=True, transform=tf)
loader = torch.utils.data.DataLoader(
    torch.utils.data.Subset(data, range(500)), batch_size=32
)

# VAE
class VAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1*28*28, 256),
            nn.ReLU()
        )
        self.mu = nn.Linear(256, 20)
        self.logvar = nn.Linear(256, 20)
        self.dec = nn.Sequential(
            nn.Linear(20, 256),
            nn.ReLU(),
            nn.Linear(256, 1*28*28),
            nn.Sigmoid()
        )

    def forward(self, x):
        h = self.enc(x)
        mu, logvar = self.mu(h), self.logvar(h)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return self.dec(z).view(-1, 1, 28, 28), mu, logvar

model = VAE()
opt = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(5):
    for x, _ in loader:
        recon, mu, logvar = model(x)
        loss = F.mse_loss(recon, x) - 0.5 * torch.mean(1 + logvar - mu**2 - logvar.exp())
        opt.zero_grad()
        loss.backward()
        opt.step()

print("VAE training completed")

100%|██████████| 9.91M/9.91M [00:00<00:00, 21.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 610kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 5.62MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 5.51MB/s]


VAE training completed
